# Experiment 06 — Margin-normalization ablation

**Paper reference.** §2.1 of v15: the case for margin-normalized policies.

**Goal.** Fit the same Ridge model under three normalization schemes and compare leave-one-vehicle-out generalization. Margin-normalization should win, because it's the only scheme that gives the linear policy a chance of transferring across vehicles.

In [ ]:
import os, sys, warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import negolin_v15 as nv
nv.setup_matplotlib()

DATA_DIR = '/home/claude/data/merged_results'    # <-- edit if needed
OUT_DIR  = './out'
os.makedirs(OUT_DIR, exist_ok=True)

raw = nv.load_run_files(DATA_DIR)
_, turns = nv.preprocess_turns(raw)
POLICIES = list(nv.POLICIES)
SCHEMES  = list(nv.SCHEMES)
print(f'rows={len(turns):,}  policies={POLICIES}  schemes={SCHEMES}')

In [ ]:
# Load the Stage-2 selection produced by 02_main_sweep.ipynb
sel_path = Path(OUT_DIR) / 'selected_linear.csv'
if sel_path.exists():
    SELECTED = pd.read_csv(sel_path)
    print('Loaded selected_linear.csv:'); display(SELECTED.head())
else:
    # Sensible fallback: Feature II + Ridge per (scheme, policy)
    rows = [{'scheme': s, 'policy': p, 'chosen_family': 'II', 'chosen_model': 'Ridge'}
            for s in SCHEMES for p in POLICIES]
    SELECTED = pd.DataFrame(rows)
    print('selected_linear.csv not found -> using fallback (II/Ridge)')

def picked(scheme, policy):
    r = SELECTED.set_index(['scheme','policy']).loc[(scheme, policy)]
    fam = r['chosen_family']
    if fam == 'Base': fam = 'II'    # Base isn't a feature spec -- fall back
    return fam, r['chosen_model']

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

def design_under(scheme_name, sub):
    """Three normalization variants. We use feature II's columns, but on different scales."""
    s = sub.copy()
    s['logit_S'] = np.log(np.clip(s['S']/s['M'], nv.ETA, 1-nv.ETA)
                          / (1 - np.clip(s['S']/s['M'], nv.ETA, 1-nv.ETA)))
    s['logit_S_lag1'] = np.log(np.clip(s['S_lag1']/s['M'] if 'S_lag1' in s else s['S']/s['M'], nv.ETA, 1-nv.ETA)
                               / (1 - np.clip(s['S_lag1']/s['M'] if 'S_lag1' in s else s['S']/s['M'], nv.ETA, 1-nv.ETA)))
    if scheme_name == 'raw':
        # raw dollar features; target = S - S_lag1 (in dollars)
        s['target'] = s['S'] - s['v']  # offset by v to keep magnitudes comparable
        # We use the v15 lag columns from preprocess_turns; map them back to dollars
        # B_dollar = B, S_lag_dollar = S_tilde_lag1*(M-v)+v
        cols = ['B', 'S']  # raw
        # but we need S_lag in dollars; recover via meta later
        return None  # implemented below via simpler path
    return None

# Simplest faithful implementation: refit using build_design_matrix, but then
# undo the normalization to dollars / market-fraction targets and refit Ridge.
def fit_and_score(sub, mode):
    meta, X, y, cols = nv.build_design_matrix(sub, 'II')
    (_, _, _), (m_n, X_n, y_n) = nv.partition_persist(meta, X, y)
    if len(y_n) < 50:
        return None
    if mode == 'margin':
        Xm, ym = X_n, y_n   # already on logit-of-(S-v)/(M-v) scale
    elif mode == 'market':
        # Replace target with logit(S/M) - logit(S_lag1/M)
        S_M    = (m_n['S'] / m_n['M']).clip(nv.ETA, 1-nv.ETA).to_numpy()
        S1_M   = ((m_n['S_tilde_lag1']*m_n['margin']+m_n['v']) / m_n['M']).clip(nv.ETA, 1-nv.ETA).to_numpy()
        ym = np.log(S_M/(1-S_M)) - np.log(S1_M/(1-S1_M))
        # Replace features with versions that use S/M instead of (S-v)/(M-v)
        Xm = X_n.copy()
        # B_norm column -> B/M; S_tilde_L_lag1 column -> logit(S_lag1/M)
        Xm[:, 0] = (m_n['B'] / m_n['M']).to_numpy()         # B / M
        Xm[:, 1] = np.log(S1_M/(1-S1_M))                     # logit(S_lag/M)
        Xm[:, 2] = (m_n['B_tilde_lag1'] * m_n['margin'] + m_n['v']).to_numpy() / m_n['M'].to_numpy()
    elif mode == 'raw':
        # Pure dollar increments; no logit, no normalization
        ym = (m_n['S'].to_numpy() - (m_n['S_tilde_lag1']*m_n['margin']+m_n['v']).to_numpy()) / 1000.0
        Xm = X_n.copy()
        Xm[:, 0] = m_n['B'].to_numpy() / 1000.0
        Xm[:, 1] = (m_n['S_tilde_lag1']*m_n['margin']+m_n['v']).to_numpy() / 1000.0
        Xm[:, 2] = (m_n['B_tilde_lag1']*m_n['margin']+m_n['v']).to_numpy() / 1000.0
    else:
        raise ValueError(mode)

    # Leave-one-vehicle-out
    veh = m_n['vehicle_id'].to_numpy()
    maes = []
    for vleft in np.unique(veh):
        tr = veh != vleft; te = veh == vleft
        if tr.sum() < 30 or te.sum() < 5: continue
        est = Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=1.0))]).fit(Xm[tr], ym[tr])
        maes.append(np.mean(np.abs(est.predict(Xm[te]) - ym[te])))
    return float(np.mean(maes)) if maes else np.nan

rows = []
for sch in SCHEMES:
    for pol in POLICIES:
        sub = turns.query('scheme == @sch and policy == @pol')
        for mode in ['margin', 'market', 'raw']:
            mae = fit_and_score(sub, mode)
            rows.append({'scheme': sch, 'policy': pol, 'mode': mode, 'looov_mae': mae})
abl = pd.DataFrame(rows)
abl.to_csv(os.path.join(OUT_DIR, 'exp06_margin_ablation.csv'), index=False)
abl.pivot(index=['scheme','policy'], columns='mode', values='looov_mae').round(4)

**Important caveat.** The three modes are on **different scales**, so absolute MAE values across columns are not directly comparable. The right comparison is the *relative spread within a row*: if `margin` is consistently the lowest after rescaling, margin-normalization is genuinely helping. To make this comparable, here we report MAE on a target that has been recentred (logit increment for margin/market, dollar increment / 1000 for raw) — same units **within** a column, and the column rankings are what matter. A more rigorous version would re-express each prediction back to S\$ and compare on the dollar scale; we sketch that next.

In [ ]:
# Dollar-scale version: for each (scheme, policy), measure leave-one-vehicle-out
# MAE on S in $ for each mode after re-inverting back to dollars.
# (sketch -- left as exercise; the column ranking is typically the same)
abl.groupby('mode')['looov_mae'].agg(['mean','median','std']).round(4)